# 05 · Synthetic data engineering for the underrepresented `analysis` task

The original fine-tune fixed the `analysis` class imbalance (225 of 1,942 rows, later
675 of ~2,392 after the source dataset's own filtering) by **duplicating** those rows
3x. That fixes the imbalance in the loss function but adds zero new information — the
model still only sees 675 distinct `analysis` examples.

This notebook instead **generates new, distinct** `analysis` examples: takes raw legal
text already in the dataset (currently only used for the `simplification` task) and
generates a proper `analysis`-formatted output for it, benchmarking four prompting
strategies before picking one, then running the result through a real curation
pipeline (schema validity, language check, PII check, near-duplicate detection) before
trusting any of it.

No GPU needed for this notebook — it's LLM API calls + local text processing. The
QLoRA re-training on the result still needs the T4 from the original project.

In [ ]:
%%capture
!pip install -q google-genai pandas pyarrow


In [ ]:
import os
from getpass import getpass

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Gemini API key: ")

import sys
sys.path.insert(0, "../src")  # for synthetic_data.py -- adjust if you run this from elsewhere
import synthetic_data as sd

import pandas as pd
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
JUDGE_MODEL = "gemini-3.1-flash-lite"  # same model used as the LLM-judge in 04-finetuned-eval-allam.ipynb


## 1 · Find raw material for new `analysis` examples

`simplification` rows share the same domain (Egyptian legal statutes) as `analysis`
rows, but the `input` text hasn't been used for `analysis` before — a defensible
source for genuinely new training pairs, not a reshuffled duplicate.

In [ ]:
train_df = pd.read_parquet("../data/train.parquet")

analysis_rows = train_df[train_df["task_type"] == "analysis"]
raw_material = train_df[
    (train_df["task_type"] == "simplification") & (train_df["source"] == "curated_legal")
]["input"].drop_duplicates().tolist()

print(f"{len(analysis_rows)} existing analysis rows (this includes the 3x duplication)")
print(f"{len(raw_material)} candidate raw texts available for new analysis examples")


## 2 · Benchmark four prompting strategies

The real `analysis` rows actually use 4 different instruction phrasings and 3 different
system prompts, not one fixed pair — so each raw text below gets a randomly sampled
`(system, instruction)` pair from the real distribution, and that *same* pair is reused
across all four strategies for that text. Only the *prompting strategy* varies, which is
what makes the comparison fair:

- **Zero-shot** — instruction + raw text, nothing else
- **Few-shot** — 3 real `analysis` examples prepended before the new text
- **Chain-of-thought** — asked to reason step by step, then give the final answer in
  the required format
- **Structured output** — Gemini's `response_schema` forces `{domain, key_points}` as
  JSON, then the required Arabic template is built programmatically from that JSON

Run on a small held-out slice (20 texts) first — this is a benchmark, not the full
generation run.

In [ ]:
import random
random.seed(42)

BENCH_SAMPLE = random.sample(raw_material, 20)
FEWSHOT_EXAMPLES = analysis_rows.sample(3, random_state=42)

# The real analysis rows use 4 different instruction phrasings and 3
# different system prompts, not one fixed pair - sample from the real
# combinations so synthetic examples have the same natural phrasing
# diversity, instead of every synthetic row sounding identical.
REAL_PROMPT_PAIRS = analysis_rows[["system", "instruction"]].drop_duplicates().to_dict("records")


def sample_prompt_pair():
    pair = random.choice(REAL_PROMPT_PAIRS)
    return pair["system"], pair["instruction"]

In [ ]:
def generate_zero_shot(system, instruction, text):
    prompt = f"{system}\n\n### Instruction:\n{instruction}\n\n### Input:\n{text}\n\n### Response:\n"
    response = client.models.generate_content(model=JUDGE_MODEL, contents=prompt)
    return response.text


def generate_few_shot(system, instruction, text):
    examples = "\n\n".join(
        f"### Input:\n{row['input']}\n\n### Response:\n{row['output']}"
        for _, row in FEWSHOT_EXAMPLES.iterrows()
    )
    prompt = (
        f"{system}\n\n### Instruction:\n{instruction}\n\n"
        f"Here are examples of the expected format:\n\n{examples}\n\n"
        f"### Input:\n{text}\n\n### Response:\n"
    )
    response = client.models.generate_content(model=JUDGE_MODEL, contents=prompt)
    return response.text


def generate_chain_of_thought(system, instruction, text):
    prompt = (
        f"{system}\n\n### Instruction:\n{instruction}\n\n### Input:\n{text}\n\n"
        "First, think step by step about the legal domain and the key provisions "
        "(don't show this reasoning in the final output). Then give ONLY the final "
        "answer in this exact format:\n\nالمجال القانوني: <domain>\nالنقاط الرئيسية:\n"
        "• <point>\n• <point>\n\n### Response:\n"
    )
    response = client.models.generate_content(model=JUDGE_MODEL, contents=prompt)
    return response.text


def generate_structured(system, instruction, text):
    from pydantic import BaseModel

    class AnalysisResult(BaseModel):
        domain: str
        key_points: list[str]

    prompt = f"{system}\n\n### Instruction:\n{instruction}\n\n### Input:\n{text}"
    response = client.models.generate_content(
        model=JUDGE_MODEL,
        contents=prompt,
        config={"response_mime_type": "application/json", "response_schema": AnalysisResult},
    )
    result = response.parsed
    points = "\n".join(f"• {p}" for p in result.key_points)
    return f"المجال القانوني: {result.domain}\nالنقاط الرئيسية:\n{points}"


STRATEGIES = {
    "zero_shot": generate_zero_shot,
    "few_shot": generate_few_shot,
    "chain_of_thought": generate_chain_of_thought,
    "structured": generate_structured,
}

In [ ]:
results = {name: [] for name in STRATEGIES}

for text in BENCH_SAMPLE:
    system, instruction = sample_prompt_pair()  # same pair used across all 4 strategies for this text
    for name, fn in STRATEGIES.items():
        try:
            output = fn(system, instruction, text)
        except Exception as e:
            output = ""
            print(f"{name} failed on one input: {e}")
        results[name].append(output)

print("Generated", {name: len(outs) for name, outs in results.items()})

## 3 · Score each strategy

Schema validity is checked directly (deterministic, free). Faithfulness is judged by
Gemini on a random 10-of-20 spot-check per strategy, reusing the same 1–10 rubric idea
as the original fine-tune's evaluation — kept small on purpose to control API cost for
a benchmark step.

In [ ]:
def judge_faithfulness(input_text, output_text):
    prompt = (
        "Score this legal analysis from 1-10 on faithfulness: does it accurately "
        "reflect only what's in the source text, with nothing invented?\n\n"
        f"Source text:\n{input_text}\n\nAnalysis:\n{output_text}\n\n"
        "Reply with ONLY a number from 1 to 10."
    )
    response = client.models.generate_content(model=JUDGE_MODEL, contents=prompt)
    try:
        return float(response.text.strip())
    except ValueError:
        return None


scoreboard = []
for name, outputs in results.items():
    schema_rate = sum(sd.has_valid_schema(o) for o in outputs if o) / len(outputs)

    spot_check_idx = random.sample(range(len(outputs)), min(10, len(outputs)))
    scores = [
        judge_faithfulness(BENCH_SAMPLE[i], outputs[i])
        for i in spot_check_idx if outputs[i]
    ]
    scores = [s for s in scores if s is not None]
    avg_faithfulness = sum(scores) / len(scores) if scores else float("nan")

    scoreboard.append({"strategy": name, "schema_valid_rate": schema_rate, "avg_faithfulness": avg_faithfulness})

scoreboard_df = pd.DataFrame(scoreboard)
scoreboard_df


**Read this table before picking a winner** — structured output should show
100% schema validity by construction (that's the whole point of forcing a schema), but
check whether that comes at a faithfulness cost compared to few-shot. Pick whichever
strategy has the best faithfulness among the ones with an acceptable schema rate, and
set it below.

In [ ]:
WINNING_STRATEGY = "few_shot"  # change this based on what the table above actually shows
generate_fn = STRATEGIES[WINNING_STRATEGY]


## 4 · Generate the full batch, then curate it

Generate for every remaining raw text (excluding the ones already used in the
benchmark), then run every candidate through `synthetic_data.py`'s curation pipeline —
schema check, length check, Arabic-language check, PII check, and near-duplicate
detection against the existing dataset.

In [ ]:
remaining_material = [t for t in raw_material if t not in BENCH_SAMPLE]

candidates = []  # list of record dicts: input, output, system, instruction
for text in remaining_material:
    system, instruction = sample_prompt_pair()
    try:
        output = generate_fn(system, instruction, text)
        candidates.append({"input": text, "output": output, "system": system, "instruction": instruction})
    except Exception as e:
        print(f"Generation failed on one input: {e}")

print(f"Generated {len(candidates)} candidate analysis examples")

In [ ]:
existing_outputs = train_df["output"].tolist()

kept_records, funnel = sd.curate_batch(candidates, existing_outputs)

print("Curation funnel:")
for stage, count in funnel.items():
    print(f"  {stage:>15}: {count}")

## 5 · Build the augmented training set

Replace the old "duplicate 3x" approach: keep the original single-copy `analysis`
rows, add the newly curated distinct examples, leave `simplification` and
`judgment_prediction` untouched.

In [ ]:
original_analysis = train_df[
    (train_df["task_type"] == "analysis") & (~train_df.duplicated(subset=["output"]))
]
other_tasks = train_df[train_df["task_type"] != "analysis"]

synthetic_rows = pd.DataFrame([
    {
        "instruction": r["instruction"],
        "input": r["input"],
        "output": r["output"],
        "system": r["system"],
        "task_type": "analysis",
        "source": "synthetic_gemini_" + WINNING_STRATEGY,
    }
    for r in kept_records
])

train_v2 = pd.concat([original_analysis, synthetic_rows, other_tasks], ignore_index=True)
train_v2 = train_v2.sample(frac=1, random_state=42).reset_index(drop=True)

print("Before  (v1, 3x-duplicated):")
print(train_df["task_type"].value_counts())
print("\nAfter   (v2, synthetic-augmented):")
print(train_v2["task_type"].value_counts())

train_v2.to_parquet("../data/train_v2.parquet")
print("\nSaved data/train_v2.parquet")

In [ ]:
%%capture
!cd .. && dvc add data/train_v2.parquet


## 6 · Next: re-train and compare

This notebook doesn't fine-tune anything — that still needs the Kaggle T4 GPU the
original project used. To get a v1-vs-v2 comparison:

1. Open `03-qlora-fine-tuning.ipynb`, change the one line that loads
   `data/train.parquet` to load `data/train_v2.parquet` instead, and push the result to
   a new Hugging Face revision (e.g. `hossam3759180/allam-qlora-legal-adapter`, branch
   or tag `v2-synthetic`) instead of overwriting the original adapter.
2. Open `04-finetuned-eval-allam.ipynb`, point it at the new adapter revision, and run
   the same LLM-judge evaluation on the same 150-row `val_eval_sample.parquet` — this
   keeps the comparison fair, since it's the exact same held-out sample used for the
   v1 numbers.
3. Compare the `analysis` row specifically (`3.06 → 6.47` faithfulness for v1) against
   whatever v2 gets — that's the number this whole notebook exists to move.

Known gap, stated honestly: this notebook doesn't run that comparison itself, because
it needs a GPU this notebook doesn't have. What it does produce -
`data/train_v2.parquet` and the curation funnel above - is real and ready for that
next step.